# 01 — Data Collection and Validation

This notebook builds the first reproducible dataset for the **Nordic Wind Decision System**.

It collects and validates:

1. DK1/DK2 settled wind generation;
2. day-ahead electricity prices;
3. imbalance prices;
4. Energinet's public wind forecast dataset;
5. exact archived weather-model runs from GFS, ICON-EU, ECMWF and DMI;
6. a leakage-audited hourly modelling panel.

## Information-time policy

The target day is nominated from information available at **11:00 Europe/Copenhagen on D-1**.

For weather data, the notebook selects the latest model run whose **assumed publication time** is no later than the decision time. The model-run initialization is exact; historical public dissemination timestamps are not fully archived, so conservative model-specific processing lags are used and retained in the audit output.

## Source documentation

- Energinet API: https://api.energidataservice.dk/index.html
- Open-Meteo Single Runs API: https://open-meteo.com/en/docs/single-runs-api
- Open-Meteo model update timing: https://open-meteo.com/en/docs/model-updates

Start with `RUN_MODE = "AUDIT"`. Do not run the full download until the audit report passes.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import time
import warnings
from datetime import date, datetime, time as dt_time, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Configuration


In [ ]:
RUN_MODE = "AUDIT"  # "AUDIT" or "FULL"

# A short, fully historical period using the post-2025 weather-run archive.
AUDIT_START_DATE = "2026-04-01"
AUDIT_END_DATE = "2026-04-04"  # exclusive

# Exact individual runs for most Open-Meteo models are archived from Sep 2025.
FULL_START_DATE = "2025-09-01"
SETTLEMENT_SAFETY_LAG_DAYS = 16

PRICE_AREAS = ["DK1", "DK2"]
LOCAL_TIMEZONE = "Europe/Copenhagen"
DECISION_LOCAL_HOUR = 11
RANDOM_SEED = 42

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "project.json"
if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as file:
        PROJECT_CONFIG = json.load(file)
else:
    PROJECT_CONFIG = {}

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = PROJECT_ROOT / "outputs" / "audits"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    AUDIT_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if RUN_MODE not in {"AUDIT", "FULL"}:
    raise ValueError("RUN_MODE must be 'AUDIT' or 'FULL'.")

if RUN_MODE == "AUDIT":
    START_DATE = AUDIT_START_DATE
    END_DATE = AUDIT_END_DATE
else:
    START_DATE = FULL_START_DATE
    settled_cutoff = (
        pd.Timestamp.now(tz=LOCAL_TIMEZONE).normalize()
        - pd.Timedelta(days=SETTLEMENT_SAFETY_LAG_DAYS)
    )
    END_DATE = settled_cutoff.date().isoformat()

LOCAL_TZ = ZoneInfo(LOCAL_TIMEZONE)
START_LOCAL = pd.Timestamp(START_DATE).tz_localize(LOCAL_TZ)
END_LOCAL = pd.Timestamp(END_DATE).tz_localize(LOCAL_TZ)
START_UTC = START_LOCAL.tz_convert("UTC")
END_UTC = END_LOCAL.tz_convert("UTC")

if END_UTC <= START_UTC:
    raise ValueError("The configured end date must be after the start date.")

EXPECTED_HOURS = len(
    pd.date_range(
        START_UTC,
        END_UTC,
        freq="h",
        inclusive="left",
    )
)
EXPECTED_TARGET_ROWS = EXPECTED_HOURS * len(PRICE_AREAS)

print("Project root:", PROJECT_ROOT)
print("Run mode:", RUN_MODE)
print("Local interval:", START_LOCAL, "to", END_LOCAL, "(exclusive)")
print("UTC interval:", START_UTC, "to", END_UTC, "(exclusive)")
print("Expected target rows:", EXPECTED_TARGET_ROWS)


## 2. Spatial weather grid and model registry

The weather points cover representative onshore, coastal and offshore locations. They are not turbine-level coordinates. Equal spatial weights are used only for the first benchmark; capacity weighting will be added later.


In [ ]:
WEATHER_POINTS = pd.DataFrame(
    [
        {"point_id": "dk1_west_jutland_north", "price_area": "DK1", "latitude": 56.70, "longitude": 8.20,  "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk1_west_jutland_south", "price_area": "DK1", "latitude": 55.55, "longitude": 8.35,  "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk1_north_jutland",       "price_area": "DK1", "latitude": 57.10, "longitude": 9.65,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_central_jutland",     "price_area": "DK1", "latitude": 56.20, "longitude": 9.20,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_south_jutland",       "price_area": "DK1", "latitude": 55.15, "longitude": 9.10,  "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_funen",               "price_area": "DK1", "latitude": 55.35, "longitude": 10.25, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk1_north_sea_west",      "price_area": "DK1", "latitude": 55.80, "longitude": 6.80,  "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk1_north_sea_north",     "price_area": "DK1", "latitude": 56.60, "longitude": 7.30,  "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk1_kattegat",            "price_area": "DK1", "latitude": 56.60, "longitude": 11.20, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_west_zealand",        "price_area": "DK2", "latitude": 55.45, "longitude": 11.25, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_east_zealand",        "price_area": "DK2", "latitude": 55.65, "longitude": 12.35, "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk2_south_zealand",       "price_area": "DK2", "latitude": 54.85, "longitude": 11.90, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_lolland_falster",     "price_area": "DK2", "latitude": 54.75, "longitude": 11.25, "environment": "coastal",  "spatial_weight": 1.0},
        {"point_id": "dk2_bornholm",            "price_area": "DK2", "latitude": 55.13, "longitude": 14.92, "environment": "onshore",  "spatial_weight": 1.0},
        {"point_id": "dk2_baltic_west",         "price_area": "DK2", "latitude": 54.70, "longitude": 12.50, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_baltic_east",         "price_area": "DK2", "latitude": 55.00, "longitude": 13.50, "environment": "offshore", "spatial_weight": 1.0},
        {"point_id": "dk2_oresund",             "price_area": "DK2", "latitude": 55.75, "longitude": 12.80, "environment": "offshore", "spatial_weight": 1.0},
    ]
)

# Conservative publication-delay proxies. These are audit assumptions, not
# claims of exact historical dissemination timestamps.
WEATHER_MODELS = {
    "gfs": {
        "api_model": "gfs_global",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "icon_eu": {
        "api_model": "icon_eu",
        "run_hours_utc": list(range(0, 24, 3)),
        "availability_lag_hours": 4,
    },
    "ecmwf": {
        "api_model": "ecmwf_ifs025",
        "run_hours_utc": [0, 6, 12, 18],
        "availability_lag_hours": 6,
    },
    "dmi": {
        "api_model": "dmi_seamless",
        "run_hours_utc": list(range(0, 24, 3)),
        "availability_lag_hours": 4,
    },
}

WEATHER_VARIABLES = [
    "wind_speed_100m",
    "wind_direction_100m",
    "wind_speed_10m",
    "temperature_2m",
    "pressure_msl",
]

display(WEATHER_POINTS)
display(pd.DataFrame(WEATHER_MODELS).T)


## 3. HTTP, caching and provenance utilities


In [ ]:
class DataDownloadError(RuntimeError):
    pass


def stable_hash(value: Any) -> str:
    encoded = json.dumps(
        value,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def request_json(
    url: str,
    params: dict[str, Any] | None = None,
    timeout: int = 180,
    max_retries: int = 6,
    pause_seconds: float = 1.0,
) -> dict[str, Any] | list[Any]:
    last_error: Exception | None = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout,
                headers={
                    "User-Agent": "nordic-wind-decision-system/0.1"
                },
            )

            if response.status_code == 429:
                retry_after = float(
                    response.headers.get(
                        "Retry-After",
                        pause_seconds * (attempt + 1),
                    )
                )
                time.sleep(max(retry_after, pause_seconds))
                continue

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == max_retries - 1:
                break
            time.sleep(pause_seconds * (2 ** attempt))

    raise DataDownloadError(
        f"Request failed after {max_retries} attempts: {url}"
    ) from last_error


def request_json_cached(
    url: str,
    params: dict[str, Any],
    cache_path: Path,
    force_refresh: bool = False,
) -> dict[str, Any] | list[Any]:
    if cache_path.exists() and not force_refresh:
        with cache_path.open("r", encoding="utf-8") as file:
            cached = json.load(file)
        return cached["payload"]

    payload = request_json(url=url, params=params)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    record = {
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "request_url": url,
        "request_params": params,
        "payload": payload,
    }

    with cache_path.open("w", encoding="utf-8") as file:
        json.dump(record, file, ensure_ascii=False)

    return payload


def trim_utc(
    frame: pd.DataFrame,
    timestamp_column: str,
    start_utc: pd.Timestamp = START_UTC,
    end_utc: pd.Timestamp = END_UTC,
) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    output = frame.copy()
    output[timestamp_column] = pd.to_datetime(
        output[timestamp_column],
        utc=True,
        errors="coerce",
    )

    return output.loc[
        output[timestamp_column].ge(start_utc)
        & output[timestamp_column].lt(end_utc)
    ].copy()


## 4. Energinet source registry and live metadata audit

The notebook records current source metadata before downloading observations. This makes dataset replacements and resolution changes visible in the Git-tracked audit outputs.


In [ ]:
EDS_BASE = "https://api.energidataservice.dk"

SOURCE_REGISTRY = pd.DataFrame(
    [
        {
            "dataset": "ProductionConsumptionSettlement",
            "role": "settlement target",
            "required": True,
            "information_time_use": "target only",
        },
        {
            "dataset": "DayAheadPrices",
            "role": "current day-ahead price",
            "required": True,
            "information_time_use": "market outcome/context",
        },
        {
            "dataset": "Elspotprices",
            "role": "historical day-ahead price fallback",
            "required": False,
            "information_time_use": "market outcome/context",
        },
        {
            "dataset": "ImbalancePrice",
            "role": "current imbalance price",
            "required": True,
            "information_time_use": "commercial outcome",
        },
        {
            "dataset": "RegulatingBalancePowerdata",
            "role": "historical imbalance fallback",
            "required": False,
            "information_time_use": "commercial outcome",
        },
        {
            "dataset": "Forecasts_Hour",
            "role": "Energinet operational wind forecast benchmark",
            "required": False,
            "information_time_use": "benchmark; revision timing to audit",
        },
    ]
)


def fetch_eds_metadata(dataset: str) -> dict[str, Any]:
    payload = request_json(f"{EDS_BASE}/meta/Dataset/{dataset}")
    if not isinstance(payload, dict):
        raise DataDownloadError(
            f"Unexpected metadata response for {dataset}."
        )
    return payload


metadata_rows: list[dict[str, Any]] = []
metadata_full: dict[str, Any] = {}

for dataset in SOURCE_REGISTRY["dataset"]:
    try:
        metadata = fetch_eds_metadata(dataset)
        metadata_full[dataset] = metadata
        metadata_rows.append(
            {
                "dataset": dataset,
                "title": metadata.get("title"),
                "resolution": metadata.get("resolution"),
                "data_from": metadata.get("dataFrom"),
                "active": metadata.get("active"),
                "last_metadata_update": metadata.get(
                    "lastMetadataUpdate"
                ),
                "metadata_status": "OK",
            }
        )
    except Exception as exc:
        metadata_rows.append(
            {
                "dataset": dataset,
                "metadata_status": f"ERROR: {exc!r}",
            }
        )

source_metadata_audit = SOURCE_REGISTRY.merge(
    pd.DataFrame(metadata_rows),
    on="dataset",
    how="left",
)

with (AUDIT_DIR / "energinet_source_metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata_full, file, indent=2, ensure_ascii=False)

source_metadata_audit.to_csv(
    AUDIT_DIR / "energinet_source_metadata_summary.csv",
    index=False,
)

display(source_metadata_audit)


## 5. Energinet download utilities


In [ ]:
def parse_eds_records(
    payload: dict[str, Any] | list[Any],
) -> list[dict[str, Any]]:
    if isinstance(payload, dict):
        for key in ["records", "result", "data"]:
            value = payload.get(key)
            if isinstance(value, list):
                return value

    if isinstance(payload, list):
        return payload

    raise DataDownloadError(
        f"Unexpected Energinet payload structure: {type(payload)}"
    )


def period_chunks(
    start: pd.Timestamp,
    end: pd.Timestamp,
    days: int,
) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    chunks: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    cursor = start

    while cursor < end:
        right = min(cursor + pd.Timedelta(days=days), end)
        chunks.append((cursor, right))
        cursor = right

    return chunks


def eds_download_chunked(
    dataset: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    filters: dict[str, list[str]] | None = None,
    chunk_days: int = 31,
    force_refresh: bool = False,
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    chunks = period_chunks(start, end, chunk_days)

    for chunk_number, (left, right) in enumerate(chunks, start=1):
        params: dict[str, Any] = {
            "start": left.isoformat(),
            "end": right.isoformat(),
            "limit": 0,
        }

        if filters:
            params["filter"] = json.dumps(
                filters,
                separators=(",", ":"),
            )

        cache_key = stable_hash(
            {
                "dataset": dataset,
                "left": left.isoformat(),
                "right": right.isoformat(),
                "filters": filters,
            }
        )
        cache_path = (
            RAW_DIR
            / "energinet"
            / dataset
            / f"{cache_key}.json"
        )

        print(
            f"[{dataset}] chunk {chunk_number}/{len(chunks)}: "
            f"{left.date()} to {right.date()}"
        )

        payload = request_json_cached(
            url=f"{EDS_BASE}/dataset/{dataset}",
            params=params,
            cache_path=cache_path,
            force_refresh=force_refresh,
        )

        records = parse_eds_records(payload)
        if records:
            part = pd.DataFrame.from_records(records)
            part["_source_dataset"] = dataset
            part["_retrieved_from_cache"] = cache_path.exists()
            parts.append(part)

        time.sleep(0.2)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def first_existing(
    columns: Iterable[str],
    candidates: Sequence[str],
) -> str:
    available = set(columns)
    for candidate in candidates:
        if candidate in available:
            return candidate

    raise KeyError(
        f"None of the candidate columns exist: {list(candidates)}"
    )


def numeric_series(
    frame: pd.DataFrame,
    column: str,
) -> pd.Series:
    return pd.to_numeric(frame[column], errors="coerce")


## 6. Settled DK1/DK2 wind generation

Settlement data are used as the **target only**. They are not treated as if they were available in real time.


In [ ]:
def resolve_wind_component(
    frame: pd.DataFrame,
    aggregate_candidates: list[str],
    detailed_candidate_sets: list[list[str]],
    component_name: str,
) -> tuple[pd.Series, list[str]]:
    for column in aggregate_candidates:
        if column in frame.columns:
            return numeric_series(frame, column), [column]

    for column_set in detailed_candidate_sets:
        if all(column in frame.columns for column in column_set):
            values = pd.concat(
                [
                    numeric_series(frame, column)
                    for column in column_set
                ],
                axis=1,
            ).sum(axis=1, min_count=1)
            return values, column_set

    raise KeyError(
        f"Could not construct {component_name}. "
        f"Available columns: {frame.columns.tolist()}"
    )


def prepare_actual_wind(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        raise ValueError(
            "ProductionConsumptionSettlement returned no records."
        )

    output = frame.copy()

    time_column = first_existing(
        output.columns,
        ["HourUTC", "TimeUTC", "QuarterUTC", "Minutes5UTC"],
    )
    area_column = first_existing(
        output.columns,
        ["PriceArea", "Area"],
    )

    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = output[area_column].astype(str)

    onshore, onshore_sources = resolve_wind_component(
        output,
        aggregate_candidates=[
            "OnshoreWindPower",
            "OnshoreWindPowerMWh",
        ],
        detailed_candidate_sets=[
            [
                "OnshoreWindGe50kW_MWh",
                "OnshoreWindLt50kW_MWh",
            ],
            [
                "OnshoreWindGe50kW",
                "OnshoreWindLt50kW",
            ],
        ],
        component_name="onshore wind",
    )

    offshore, offshore_sources = resolve_wind_component(
        output,
        aggregate_candidates=[
            "OffshoreWindPower",
            "OffshoreWindPowerMWh",
        ],
        detailed_candidate_sets=[
            [
                "OffshoreWindGe100MW_MWh",
                "OffshoreWindLt100MW_MWh",
            ],
            [
                "OffshoreWindGe100MW",
                "OffshoreWindLt100MW",
            ],
        ],
        component_name="offshore wind",
    )

    output["actual_onshore_wind_mwh"] = onshore
    output["actual_offshore_wind_mwh"] = offshore
    output["actual_wind_mwh"] = onshore + offshore
    output["wind_source_columns"] = (
        "onshore="
        + ",".join(onshore_sources)
        + "; offshore="
        + ",".join(offshore_sources)
    )

    result = output[
        [
            "timestamp_utc",
            "price_area",
            "actual_onshore_wind_mwh",
            "actual_offshore_wind_mwh",
            "actual_wind_mwh",
            "wind_source_columns",
        ]
    ].copy()

    result = trim_utc(result, "timestamp_utc")

    return (
        result.sort_values(["timestamp_utc", "price_area"])
        .drop_duplicates(
            ["timestamp_utc", "price_area"],
            keep="last",
        )
        .reset_index(drop=True)
    )


production_raw = eds_download_chunked(
    dataset="ProductionConsumptionSettlement",
    start=START_UTC - pd.Timedelta(days=1),
    end=END_UTC + pd.Timedelta(days=1),
    filters={"PriceArea": PRICE_AREAS},
    chunk_days=7 if RUN_MODE == "AUDIT" else 31,
)

actual_wind = prepare_actual_wind(production_raw)

print("Actual wind shape:", actual_wind.shape)
display(actual_wind.head())
display(
    actual_wind.groupby("price_area")[
        "actual_wind_mwh"
    ].describe()
)


## 7. Day-ahead prices

Both the current and historical Energinet datasets are attempted. Overlapping records prefer `DayAheadPrices`.


In [ ]:
PRICE_TIME_CANDIDATES = [
    "TimeUTC",
    "HourUTC",
    "QuarterUTC",
    "Minutes15UTC",
    "Minutes5UTC",
]

PRICE_VALUE_CANDIDATES = [
    "DayAheadPriceEUR",
    "SpotPriceEUR",
    "PriceEUR",
    "DayAheadPrice",
    "SpotPrice",
]


def prepare_day_ahead_prices(
    frame: pd.DataFrame,
    source_dataset: str,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    time_column = first_existing(
        frame.columns,
        PRICE_TIME_CANDIDATES,
    )
    area_column = first_existing(
        frame.columns,
        ["PriceArea", "Area"],
    )
    price_column = first_existing(
        frame.columns,
        PRICE_VALUE_CANDIDATES,
    )

    output = frame.copy()
    output["raw_timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["timestamp_utc"] = (
        output["raw_timestamp_utc"].dt.floor("h")
    )
    output["price_area"] = output[area_column].astype(str)
    output["raw_price_eur_mwh"] = pd.to_numeric(
        output[price_column],
        errors="coerce",
    )
    output["price_source_dataset"] = source_dataset

    hourly = (
        output.groupby(
            [
                "timestamp_utc",
                "price_area",
                "price_source_dataset",
            ],
            as_index=False,
        )
        .agg(
            day_ahead_price_eur_mwh=(
                "raw_price_eur_mwh",
                "mean",
            ),
            day_ahead_intervals_per_hour=(
                "raw_timestamp_utc",
                "nunique",
            ),
        )
    )

    return trim_utc(hourly, "timestamp_utc")


price_parts: list[pd.DataFrame] = []
price_errors: list[dict[str, str]] = []

for dataset in ["Elspotprices", "DayAheadPrices"]:
    try:
        raw = eds_download_chunked(
            dataset=dataset,
            start=START_UTC - pd.Timedelta(days=1),
            end=END_UTC + pd.Timedelta(days=1),
            filters={"PriceArea": PRICE_AREAS},
            chunk_days=7 if RUN_MODE == "AUDIT" else 31,
        )
        prepared = prepare_day_ahead_prices(raw, dataset)
        if not prepared.empty:
            price_parts.append(prepared)
    except Exception as exc:
        price_errors.append(
            {"dataset": dataset, "error": repr(exc)}
        )

if not price_parts:
    raise ValueError(
        "No usable day-ahead price records were downloaded."
    )

day_ahead_prices = pd.concat(
    price_parts,
    ignore_index=True,
)

source_priority = {
    "Elspotprices": 1,
    "DayAheadPrices": 2,
}

day_ahead_prices["_source_priority"] = (
    day_ahead_prices["price_source_dataset"]
    .map(source_priority)
    .fillna(0)
)

day_ahead_prices = (
    day_ahead_prices.sort_values(
        [
            "timestamp_utc",
            "price_area",
            "_source_priority",
        ]
    )
    .drop_duplicates(
        ["timestamp_utc", "price_area"],
        keep="last",
    )
    .drop(columns="_source_priority")
    .reset_index(drop=True)
)

print("Day-ahead price shape:", day_ahead_prices.shape)
display(day_ahead_prices.head())
display(pd.DataFrame(price_errors))


## 8. Imbalance prices

The native-resolution table is preserved. A separate hourly mean is produced only for the first hourly modelling panel.


In [ ]:
IMBALANCE_TIME_CANDIDATES = [
    "TimeUTC",
    "QuarterUTC",
    "HourUTC",
    "Minutes15UTC",
    "Minutes5UTC",
]

IMBALANCE_VALUE_CANDIDATES = [
    "ImbalancePriceEUR",
    "ImbalancePrice",
    "BalancePowerPriceEUR",
    "BalancingPowerPriceEUR",
]


def prepare_imbalance_prices(
    frame: pd.DataFrame,
    source_dataset: str,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    time_column = first_existing(
        frame.columns,
        IMBALANCE_TIME_CANDIDATES,
    )
    area_column = first_existing(
        frame.columns,
        ["PriceArea", "Area"],
    )
    value_column = first_existing(
        frame.columns,
        IMBALANCE_VALUE_CANDIDATES,
    )

    output = frame.copy()
    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = output[area_column].astype(str)
    output["imbalance_price_eur_mwh"] = pd.to_numeric(
        output[value_column],
        errors="coerce",
    )
    output["imbalance_source_dataset"] = source_dataset

    keep_columns = [
        "timestamp_utc",
        "price_area",
        "imbalance_price_eur_mwh",
        "imbalance_source_dataset",
    ]

    return output[keep_columns].dropna(
        subset=["timestamp_utc"]
    )


imbalance_parts: list[pd.DataFrame] = []
imbalance_errors: list[dict[str, str]] = []

for dataset in [
    "RegulatingBalancePowerdata",
    "ImbalancePrice",
]:
    try:
        raw = eds_download_chunked(
            dataset=dataset,
            start=START_UTC - pd.Timedelta(days=1),
            end=END_UTC + pd.Timedelta(days=1),
            filters={"PriceArea": PRICE_AREAS},
            chunk_days=7 if RUN_MODE == "AUDIT" else 31,
        )
        prepared = prepare_imbalance_prices(raw, dataset)
        if not prepared.empty:
            imbalance_parts.append(prepared)
    except Exception as exc:
        imbalance_errors.append(
            {"dataset": dataset, "error": repr(exc)}
        )

if imbalance_parts:
    imbalance_native = pd.concat(
        imbalance_parts,
        ignore_index=True,
    )
    imbalance_native = trim_utc(
        imbalance_native,
        "timestamp_utc",
    )

    imbalance_priority = {
        "RegulatingBalancePowerdata": 1,
        "ImbalancePrice": 2,
    }
    imbalance_native["_source_priority"] = (
        imbalance_native["imbalance_source_dataset"]
        .map(imbalance_priority)
        .fillna(0)
    )
    imbalance_native = (
        imbalance_native.sort_values(
            [
                "timestamp_utc",
                "price_area",
                "_source_priority",
            ]
        )
        .drop_duplicates(
            ["timestamp_utc", "price_area"],
            keep="last",
        )
        .drop(columns="_source_priority")
        .reset_index(drop=True)
    )

    imbalance_hourly = imbalance_native.copy()
    imbalance_hourly["hour_utc"] = (
        imbalance_hourly["timestamp_utc"].dt.floor("h")
    )
    imbalance_hourly = (
        imbalance_hourly.groupby(
            ["hour_utc", "price_area"],
            as_index=False,
        )
        .agg(
            imbalance_price_eur_mwh=(
                "imbalance_price_eur_mwh",
                "mean",
            ),
            imbalance_intervals_per_hour=(
                "timestamp_utc",
                "nunique",
            ),
            imbalance_source_dataset=(
                "imbalance_source_dataset",
                "last",
            ),
        )
        .rename(columns={"hour_utc": "timestamp_utc"})
    )
else:
    imbalance_native = pd.DataFrame()
    imbalance_hourly = pd.DataFrame()

print("Native imbalance shape:", imbalance_native.shape)
print("Hourly imbalance shape:", imbalance_hourly.shape)
display(imbalance_native.head())
display(pd.DataFrame(imbalance_errors))


## 9. Energinet public wind forecast benchmark

This source is downloaded and retained, but it is **not yet labelled tradable**. The historical revision/publication structure must be verified before using it as a strict ex-ante benchmark.


In [ ]:
def prepare_energinet_wind_forecast(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    output = frame.copy()

    time_column = first_existing(
        output.columns,
        [
            "HourUTC",
            "TimeUTC",
            "Minutes5UTC",
            "QuarterUTC",
        ],
    )
    area_column = first_existing(
        output.columns,
        ["PriceArea", "Area"],
    )

    output["timestamp_utc"] = pd.to_datetime(
        output[time_column],
        utc=True,
        errors="coerce",
    )
    output["price_area"] = output[area_column].astype(str)

    numeric_candidates = [
        column
        for column in output.columns
        if "wind" in column.lower()
        and "solar" not in column.lower()
    ]

    for column in numeric_candidates:
        output[column] = pd.to_numeric(
            output[column],
            errors="coerce",
        )

    preferred_total_candidates = [
        "WindPower",
        "WindPowerForecast",
        "ForecastWindPower",
        "TotalWindPower",
        "WindPowerMWh",
    ]

    total_column = next(
        (
            column
            for column in preferred_total_candidates
            if column in output.columns
        ),
        None,
    )

    if total_column is not None:
        output["energinet_wind_forecast_mw"] = output[
            total_column
        ]
        source_columns = [total_column]
    else:
        component_columns = [
            column
            for column in numeric_candidates
            if any(
                token in column.lower()
                for token in ["onshore", "offshore"]
            )
        ]

        if not component_columns:
            raise KeyError(
                "Could not identify wind forecast columns. "
                f"Available columns: {output.columns.tolist()}"
            )

        output["energinet_wind_forecast_mw"] = output[
            component_columns
        ].sum(axis=1, min_count=1)
        source_columns = component_columns

    result = output[
        [
            "timestamp_utc",
            "price_area",
            "energinet_wind_forecast_mw",
        ]
    ].copy()
    result["energinet_forecast_source_columns"] = (
        ",".join(source_columns)
    )
    result = trim_utc(result, "timestamp_utc")

    return (
        result.groupby(
            ["timestamp_utc", "price_area"],
            as_index=False,
        )
        .agg(
            energinet_wind_forecast_mw=(
                "energinet_wind_forecast_mw",
                "last",
            ),
            energinet_forecast_source_columns=(
                "energinet_forecast_source_columns",
                "last",
            ),
        )
    )


try:
    energinet_forecast_raw = eds_download_chunked(
        dataset="Forecasts_Hour",
        start=START_UTC - pd.Timedelta(days=1),
        end=END_UTC + pd.Timedelta(days=1),
        filters={"PriceArea": PRICE_AREAS},
        chunk_days=7 if RUN_MODE == "AUDIT" else 31,
    )
    energinet_wind_forecast = (
        prepare_energinet_wind_forecast(
            energinet_forecast_raw
        )
    )
    energinet_forecast_error = None
except Exception as exc:
    energinet_forecast_raw = pd.DataFrame()
    energinet_wind_forecast = pd.DataFrame()
    energinet_forecast_error = repr(exc)

print(
    "Energinet wind forecast shape:",
    energinet_wind_forecast.shape,
)
print("Optional-source error:", energinet_forecast_error)
display(energinet_wind_forecast.head())


## 10. Exact weather-run manifest

For each target day and model, the notebook selects the latest run satisfying:

`run initialization + conservative availability lag <= decision time`.


In [ ]:
def latest_eligible_run(
    decision_time_utc: pd.Timestamp,
    run_hours_utc: list[int],
    availability_lag_hours: int,
) -> pd.Timestamp:
    cutoff = (
        decision_time_utc
        - pd.Timedelta(hours=availability_lag_hours)
    )

    candidates: list[pd.Timestamp] = []

    for day_offset in range(0, 3):
        candidate_date = (
            cutoff.normalize()
            - pd.Timedelta(days=day_offset)
        )

        for run_hour in run_hours_utc:
            candidate = (
                candidate_date
                + pd.Timedelta(hours=run_hour)
            )
            if candidate <= cutoff:
                candidates.append(candidate)

    if not candidates:
        raise ValueError(
            f"No eligible run found before {decision_time_utc}."
        )

    return max(candidates)


def build_weather_run_manifest() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    target_dates = pd.date_range(
        START_LOCAL.normalize(),
        END_LOCAL.normalize(),
        freq="D",
        inclusive="left",
    )

    for target_start_local in target_dates:
        target_date_local = target_start_local.date()
        decision_date = target_date_local - timedelta(days=1)
        decision_time_local = pd.Timestamp(
            datetime.combine(
                decision_date,
                dt_time(
                    hour=DECISION_LOCAL_HOUR,
                    minute=0,
                ),
                tzinfo=LOCAL_TZ,
            )
        )
        decision_time_utc = decision_time_local.tz_convert("UTC")

        target_end_local = (
            target_start_local + pd.DateOffset(days=1)
        )

        for model_alias, specification in WEATHER_MODELS.items():
            run_init_utc = latest_eligible_run(
                decision_time_utc=decision_time_utc,
                run_hours_utc=specification[
                    "run_hours_utc"
                ],
                availability_lag_hours=specification[
                    "availability_lag_hours"
                ],
            )
            assumed_available_utc = (
                run_init_utc
                + pd.Timedelta(
                    hours=specification[
                        "availability_lag_hours"
                    ]
                )
            )

            rows.append(
                {
                    "target_date_local": (
                        target_date_local.isoformat()
                    ),
                    "model_alias": model_alias,
                    "api_model": specification["api_model"],
                    "decision_time_local": (
                        decision_time_local.isoformat()
                    ),
                    "decision_time_utc": (
                        decision_time_utc.isoformat()
                    ),
                    "run_init_utc": (
                        run_init_utc.isoformat()
                    ),
                    "assumed_available_utc": (
                        assumed_available_utc.isoformat()
                    ),
                    "availability_lag_hours": specification[
                        "availability_lag_hours"
                    ],
                    "target_start_utc": (
                        target_start_local
                        .tz_convert("UTC")
                        .isoformat()
                    ),
                    "target_end_utc": (
                        target_end_local
                        .tz_convert("UTC")
                        .isoformat()
                    ),
                    "information_time_safe": (
                        assumed_available_utc
                        <= decision_time_utc
                    ),
                }
            )

    manifest = pd.DataFrame(rows)

    for column in [
        "decision_time_utc",
        "run_init_utc",
        "assumed_available_utc",
        "target_start_utc",
        "target_end_utc",
    ]:
        manifest[column] = pd.to_datetime(
            manifest[column],
            utc=True,
        )

    return manifest


weather_run_manifest = build_weather_run_manifest()
weather_run_manifest.to_csv(
    AUDIT_DIR / "weather_run_manifest.csv",
    index=False,
)

display(weather_run_manifest)


## 11. Download exact Open-Meteo single runs

Responses are cached by model and run initialization, so interrupted full downloads can resume.


In [ ]:
OPEN_METEO_SINGLE_RUN_URL = (
    "https://single-runs-api.open-meteo.com/v1/forecast"
)


def parse_open_meteo_locations(
    payload: dict[str, Any] | list[Any],
    points: pd.DataFrame,
    manifest_row: pd.Series,
) -> pd.DataFrame:
    payload_items = payload if isinstance(payload, list) else [payload]

    if len(payload_items) != len(points):
        raise DataDownloadError(
            "Open-Meteo returned a different number of "
            f"locations: requested={len(points)}, "
            f"returned={len(payload_items)}"
        )

    frames: list[pd.DataFrame] = []

    for (_, point), location_payload in zip(
        points.reset_index(drop=True).iterrows(),
        payload_items,
    ):
        if not isinstance(location_payload, dict):
            raise DataDownloadError(
                "Unexpected Open-Meteo location payload."
            )

        hourly = location_payload.get("hourly")
        if not isinstance(hourly, dict) or "time" not in hourly:
            raise DataDownloadError(
                f"No hourly data for {manifest_row['model_alias']} "
                f"at {point['point_id']}."
            )

        frame = pd.DataFrame(hourly)
        frame["timestamp_utc"] = pd.to_datetime(
            frame.pop("time"),
            utc=True,
            errors="coerce",
        )
        frame["model_alias"] = manifest_row["model_alias"]
        frame["api_model"] = manifest_row["api_model"]
        frame["run_init_utc"] = manifest_row["run_init_utc"]
        frame["assumed_available_utc"] = manifest_row[
            "assumed_available_utc"
        ]
        frame["decision_time_utc"] = manifest_row[
            "decision_time_utc"
        ]
        frame["target_date_local"] = manifest_row[
            "target_date_local"
        ]
        frame["point_id"] = point["point_id"]
        frame["price_area"] = point["price_area"]
        frame["environment"] = point["environment"]
        frame["spatial_weight"] = point["spatial_weight"]
        frame["requested_latitude"] = point["latitude"]
        frame["requested_longitude"] = point["longitude"]
        frame["returned_latitude"] = location_payload.get(
            "latitude"
        )
        frame["returned_longitude"] = location_payload.get(
            "longitude"
        )
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def download_single_run(
    manifest_row: pd.Series,
    points: pd.DataFrame,
    force_refresh: bool = False,
) -> pd.DataFrame:
    run_init_utc = pd.Timestamp(
        manifest_row["run_init_utc"]
    )
    target_end_utc = pd.Timestamp(
        manifest_row["target_end_utc"]
    )

    forecast_hours = int(
        math.ceil(
            (
                target_end_utc - run_init_utc
            )
            / pd.Timedelta(hours=1)
        )
    ) + 2

    params = {
        "latitude": ",".join(
            points["latitude"].astype(str)
        ),
        "longitude": ",".join(
            points["longitude"].astype(str)
        ),
        "hourly": ",".join(WEATHER_VARIABLES),
        "models": manifest_row["api_model"],
        "run": run_init_utc.strftime("%Y-%m-%dT%H:%M"),
        "forecast_hours": forecast_hours,
        "timezone": "UTC",
        "wind_speed_unit": "ms",
        "timeformat": "iso8601",
    }

    cache_path = (
        RAW_DIR
        / "open_meteo_single_runs"
        / str(manifest_row["model_alias"])
        / (
            run_init_utc.strftime("%Y%m%dT%H%MZ")
            + ".json"
        )
    )

    payload = request_json_cached(
        url=OPEN_METEO_SINGLE_RUN_URL,
        params=params,
        cache_path=cache_path,
        force_refresh=force_refresh,
    )

    parsed = parse_open_meteo_locations(
        payload=payload,
        points=points,
        manifest_row=manifest_row,
    )

    target_start_utc = pd.Timestamp(
        manifest_row["target_start_utc"]
    )
    target_end_utc = pd.Timestamp(
        manifest_row["target_end_utc"]
    )

    return parsed.loc[
        parsed["timestamp_utc"].ge(target_start_utc)
        & parsed["timestamp_utc"].lt(target_end_utc)
    ].copy()


weather_parts: list[pd.DataFrame] = []
weather_errors: list[dict[str, Any]] = []

for row_number, manifest_row in weather_run_manifest.iterrows():
    print(
        f"[weather] {row_number + 1}/"
        f"{len(weather_run_manifest)} "
        f"{manifest_row['model_alias']} "
        f"{manifest_row['target_date_local']} "
        f"run={manifest_row['run_init_utc']}"
    )

    try:
        part = download_single_run(
            manifest_row=manifest_row,
            points=WEATHER_POINTS,
        )
        weather_parts.append(part)
    except Exception as exc:
        weather_errors.append(
            {
                "target_date_local": manifest_row[
                    "target_date_local"
                ],
                "model_alias": manifest_row["model_alias"],
                "api_model": manifest_row["api_model"],
                "run_init_utc": manifest_row[
                    "run_init_utc"
                ],
                "error": repr(exc),
            }
        )

if weather_parts:
    weather_point_forecasts = pd.concat(
        weather_parts,
        ignore_index=True,
    )
    weather_point_forecasts = (
        weather_point_forecasts.sort_values(
            [
                "timestamp_utc",
                "price_area",
                "model_alias",
                "point_id",
            ]
        )
        .drop_duplicates(
            [
                "timestamp_utc",
                "model_alias",
                "point_id",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )
else:
    weather_point_forecasts = pd.DataFrame()

weather_error_report = pd.DataFrame(weather_errors)
weather_error_report.to_csv(
    AUDIT_DIR / "weather_download_errors.csv",
    index=False,
)

print(
    "Point-level weather shape:",
    weather_point_forecasts.shape,
)
display(weather_point_forecasts.head())
display(weather_error_report)


## 12. Aggregate point forecasts to DK1/DK2 model channels


In [ ]:
def weighted_average(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    valid = values.notna() & weights.notna()

    if not valid.any():
        return np.nan

    return float(
        np.average(
            values.loc[valid],
            weights=weights.loc[valid],
        )
    )


def aggregate_weather_by_area(
    point_forecasts: pd.DataFrame,
) -> pd.DataFrame:
    if point_forecasts.empty:
        return pd.DataFrame()

    working = point_forecasts.copy()

    for variable in WEATHER_VARIABLES:
        working[variable] = pd.to_numeric(
            working[variable],
            errors="coerce",
        )

    direction_radians = np.deg2rad(
        working["wind_direction_100m"]
    )
    working["wind_direction_100m_sin"] = np.sin(
        direction_radians
    )
    working["wind_direction_100m_cos"] = np.cos(
        direction_radians
    )
    working["wind_speed_100m_cubed"] = (
        working["wind_speed_100m"].clip(lower=0) ** 3
    )
    working["lead_hours"] = (
        working["timestamp_utc"]
        - working["run_init_utc"]
    ) / pd.Timedelta(hours=1)

    value_columns = [
        "wind_speed_100m",
        "wind_speed_100m_cubed",
        "wind_direction_100m_sin",
        "wind_direction_100m_cos",
        "wind_speed_10m",
        "temperature_2m",
        "pressure_msl",
        "lead_hours",
    ]

    group_columns = [
        "timestamp_utc",
        "price_area",
        "model_alias",
        "api_model",
        "run_init_utc",
        "assumed_available_utc",
        "decision_time_utc",
        "target_date_local",
    ]

    rows: list[dict[str, Any]] = []

    for keys, group in working.groupby(
        group_columns,
        sort=False,
    ):
        row = dict(zip(group_columns, keys))

        masks = {
            "all": pd.Series(True, index=group.index),
            "land": group["environment"].isin(
                ["onshore", "coastal"]
            ),
            "offshore": group["environment"].eq(
                "offshore"
            ),
        }

        for region_name, mask in masks.items():
            subset = group.loc[mask]
            row[f"points_{region_name}"] = (
                subset["point_id"].nunique()
            )

            for column in value_columns:
                row[f"{column}_{region_name}"] = (
                    weighted_average(
                        subset[column],
                        subset["spatial_weight"],
                    )
                )

        row["information_time_safe"] = bool(
            row["assumed_available_utc"]
            <= row["decision_time_utc"]
        )
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "timestamp_utc",
                "price_area",
                "model_alias",
            ]
        )
        .reset_index(drop=True)
    )


weather_area_long = aggregate_weather_by_area(
    weather_point_forecasts
)

print("Area-level weather shape:", weather_area_long.shape)
display(weather_area_long.head())


## 13. Build the hourly modelling panel

The panel contains outcomes and ex-ante weather channels. Recent settled generation lags are deliberately **not** added here.


In [ ]:
def weather_long_to_wide(
    weather_long: pd.DataFrame,
) -> pd.DataFrame:
    if weather_long.empty:
        return pd.DataFrame(
            columns=["timestamp_utc", "price_area"]
        )

    identifier_columns = {
        "timestamp_utc",
        "price_area",
        "model_alias",
        "api_model",
        "run_init_utc",
        "assumed_available_utc",
        "decision_time_utc",
        "target_date_local",
        "information_time_safe",
    }

    feature_columns = [
        column
        for column in weather_long.columns
        if column not in identifier_columns
    ]

    wide_parts: list[pd.DataFrame] = []

    for feature in feature_columns:
        pivot = weather_long.pivot_table(
            index=["timestamp_utc", "price_area"],
            columns="model_alias",
            values=feature,
            aggfunc="first",
        )
        pivot.columns = [
            f"{model_alias}__{feature}"
            for model_alias in pivot.columns
        ]
        wide_parts.append(pivot)

    return pd.concat(
        wide_parts,
        axis=1,
    ).reset_index()


weather_wide = weather_long_to_wide(
    weather_area_long
)

model_panel = actual_wind.merge(
    day_ahead_prices[
        [
            "timestamp_utc",
            "price_area",
            "day_ahead_price_eur_mwh",
            "day_ahead_intervals_per_hour",
            "price_source_dataset",
        ]
    ],
    on=["timestamp_utc", "price_area"],
    how="left",
    validate="one_to_one",
)

if not imbalance_hourly.empty:
    model_panel = model_panel.merge(
        imbalance_hourly,
        on=["timestamp_utc", "price_area"],
        how="left",
        validate="one_to_one",
    )

if not energinet_wind_forecast.empty:
    model_panel = model_panel.merge(
        energinet_wind_forecast,
        on=["timestamp_utc", "price_area"],
        how="left",
        validate="one_to_one",
    )

if not weather_wide.empty:
    model_panel = model_panel.merge(
        weather_wide,
        on=["timestamp_utc", "price_area"],
        how="left",
        validate="one_to_one",
    )

model_panel = model_panel.sort_values(
    ["price_area", "timestamp_utc"]
).reset_index(drop=True)

local_timestamp = model_panel[
    "timestamp_utc"
].dt.tz_convert(LOCAL_TIMEZONE)

model_panel["target_date_local"] = (
    local_timestamp.dt.date.astype(str)
)
model_panel["hour_local"] = local_timestamp.dt.hour
model_panel["hour_utc"] = (
    model_panel["timestamp_utc"].dt.hour
)
model_panel["day_of_week"] = local_timestamp.dt.dayofweek
model_panel["month"] = local_timestamp.dt.month
model_panel["is_weekend"] = (
    model_panel["day_of_week"].ge(5).astype(int)
)
model_panel["hour_local_sin"] = np.sin(
    2 * np.pi * model_panel["hour_local"] / 24
)
model_panel["hour_local_cos"] = np.cos(
    2 * np.pi * model_panel["hour_local"] / 24
)
model_panel["month_sin"] = np.sin(
    2 * np.pi * model_panel["month"] / 12
)
model_panel["month_cos"] = np.cos(
    2 * np.pi * model_panel["month"] / 12
)

if {
    "day_ahead_price_eur_mwh",
    "imbalance_price_eur_mwh",
}.issubset(model_panel.columns):
    model_panel["imbalance_spread_eur_mwh"] = (
        model_panel["imbalance_price_eur_mwh"]
        - model_panel["day_ahead_price_eur_mwh"]
    )

print("Model panel shape:", model_panel.shape)
display(model_panel.head())


## 14. Leakage and data-quality audit


In [ ]:
def check_row(
    check: str,
    passed: bool,
    value: Any,
    threshold: Any,
    severity: str = "BLOCKING",
    notes: str = "",
) -> dict[str, Any]:
    return {
        "check": check,
        "status": "PASS" if passed else "FAIL",
        "value": value,
        "threshold": threshold,
        "severity": severity,
        "notes": notes,
    }


checks: list[dict[str, Any]] = []

checks.append(
    check_row(
        check="actual wind row count",
        passed=len(actual_wind) == EXPECTED_TARGET_ROWS,
        value=len(actual_wind),
        threshold=EXPECTED_TARGET_ROWS,
    )
)

checks.append(
    check_row(
        check="actual wind duplicate rate",
        passed=not actual_wind.duplicated(
            ["timestamp_utc", "price_area"]
        ).any(),
        value=float(
            actual_wind.duplicated(
                ["timestamp_utc", "price_area"]
            ).mean()
        ),
        threshold=0,
    )
)

checks.append(
    check_row(
        check="actual wind missing target rate",
        passed=actual_wind["actual_wind_mwh"]
        .isna()
        .mean()
        <= 0.001,
        value=float(
            actual_wind["actual_wind_mwh"]
            .isna()
            .mean()
        ),
        threshold="<= 0.1%",
    )
)

checks.append(
    check_row(
        check="day-ahead price coverage",
        passed=day_ahead_prices[
            "day_ahead_price_eur_mwh"
        ].notna().mean()
        >= 0.99,
        value=float(
            day_ahead_prices[
                "day_ahead_price_eur_mwh"
            ].notna().mean()
        ),
        threshold=">= 99%",
    )
)

safe_manifest_rate = float(
    weather_run_manifest[
        "information_time_safe"
    ].mean()
)

checks.append(
    check_row(
        check="weather run information-time safety",
        passed=safe_manifest_rate == 1.0,
        value=safe_manifest_rate,
        threshold=1.0,
    )
)

successful_weather_models = (
    weather_area_long["model_alias"].nunique()
    if not weather_area_long.empty
    else 0
)

checks.append(
    check_row(
        check="successful exact weather models",
        passed=successful_weather_models >= 3,
        value=successful_weather_models,
        threshold=">= 3",
    )
)

weather_feature_columns = [
    column
    for column in model_panel.columns
    if "__wind_speed_100m_all" in column
]

for column in weather_feature_columns:
    coverage = float(
        model_panel[column].notna().mean()
    )
    checks.append(
        check_row(
            check=f"coverage: {column}",
            passed=coverage >= 0.95,
            value=coverage,
            threshold=">= 95%",
        )
    )

if not imbalance_hourly.empty:
    imbalance_coverage = float(
        model_panel["imbalance_price_eur_mwh"]
        .notna()
        .mean()
    )
    checks.append(
        check_row(
            check="imbalance price coverage",
            passed=imbalance_coverage >= 0.95,
            value=imbalance_coverage,
            threshold=">= 95%",
            severity="WARNING",
            notes=(
                "Commercial evaluation can proceed only "
                "on common support."
            ),
        )
    )

checks.append(
    check_row(
        check="Energinet benchmark parsed",
        passed=not energinet_wind_forecast.empty,
        value=len(energinet_wind_forecast),
        threshold="non-empty",
        severity="WARNING",
        notes=(
            "Revision timing remains unverified even if "
            "the source parses successfully."
        ),
    )
)

quality_report = pd.DataFrame(checks)

blocking_failures = quality_report.loc[
    quality_report["status"].eq("FAIL")
    & quality_report["severity"].eq("BLOCKING")
]

display(quality_report)
print("Blocking failures:", len(blocking_failures))


## 15. Visual sanity checks


In [ ]:
for price_area in PRICE_AREAS:
    area_frame = actual_wind.loc[
        actual_wind["price_area"].eq(price_area)
    ]

    figure, axis = plt.subplots(figsize=(14, 4))
    axis.plot(
        area_frame["timestamp_utc"],
        area_frame["actual_wind_mwh"],
        label="Actual wind",
    )

    if not energinet_wind_forecast.empty:
        benchmark = energinet_wind_forecast.loc[
            energinet_wind_forecast[
                "price_area"
            ].eq(price_area)
        ]
        axis.plot(
            benchmark["timestamp_utc"],
            benchmark[
                "energinet_wind_forecast_mw"
            ],
            label="Energinet forecast",
            alpha=0.8,
        )

    axis.set_title(
        f"{price_area}: wind generation and public forecast"
    )
    axis.set_ylabel("MW / MWh per hour")
    axis.set_xlabel("UTC")
    axis.legend()
    figure.tight_layout()

    output_path = (
        FIGURE_DIR
        / f"notebook_01_{price_area.lower()}_wind.png"
    )
    figure.savefig(output_path, dpi=150)
    plt.show()


if not weather_area_long.empty:
    for price_area in PRICE_AREAS:
        plot_frame = (
            weather_area_long.loc[
                weather_area_long["price_area"].eq(
                    price_area
                )
            ]
            .pivot_table(
                index="timestamp_utc",
                columns="model_alias",
                values="wind_speed_100m_all",
                aggfunc="first",
            )
            .sort_index()
        )

        axis = plot_frame.plot(
            figsize=(14, 5),
            title=(
                f"{price_area}: exact-run 100 m "
                "wind-speed forecasts"
            ),
        )
        axis.set_ylabel("m/s")
        axis.set_xlabel("UTC")
        plt.tight_layout()
        plt.savefig(
            FIGURE_DIR
            / (
                f"notebook_01_"
                f"{price_area.lower()}_weather.png"
            ),
            dpi=150,
        )
        plt.show()


## 16. Save validated outputs and verification report


In [ ]:
OUTPUT_FILES = {
    "actual_wind": (
        INTERIM_DIR / "actual_wind_hourly.parquet"
    ),
    "day_ahead_prices": (
        INTERIM_DIR / "day_ahead_prices_hourly.parquet"
    ),
    "imbalance_native": (
        INTERIM_DIR / "imbalance_prices_native.parquet"
    ),
    "imbalance_hourly": (
        INTERIM_DIR / "imbalance_prices_hourly.parquet"
    ),
    "energinet_forecast": (
        INTERIM_DIR
        / "energinet_wind_forecast_hourly.parquet"
    ),
    "weather_point": (
        INTERIM_DIR
        / "weather_exact_runs_point.parquet"
    ),
    "weather_area_long": (
        INTERIM_DIR
        / "weather_exact_runs_area_long.parquet"
    ),
    "model_panel": (
        PROCESSED_DIR
        / "wind_model_panel_hourly.parquet"
    ),
}

actual_wind.to_parquet(
    OUTPUT_FILES["actual_wind"],
    index=False,
)
day_ahead_prices.to_parquet(
    OUTPUT_FILES["day_ahead_prices"],
    index=False,
)

if not imbalance_native.empty:
    imbalance_native.to_parquet(
        OUTPUT_FILES["imbalance_native"],
        index=False,
    )

if not imbalance_hourly.empty:
    imbalance_hourly.to_parquet(
        OUTPUT_FILES["imbalance_hourly"],
        index=False,
    )

if not energinet_wind_forecast.empty:
    energinet_wind_forecast.to_parquet(
        OUTPUT_FILES["energinet_forecast"],
        index=False,
    )

if not weather_point_forecasts.empty:
    weather_point_forecasts.to_parquet(
        OUTPUT_FILES["weather_point"],
        index=False,
    )

if not weather_area_long.empty:
    weather_area_long.to_parquet(
        OUTPUT_FILES["weather_area_long"],
        index=False,
    )

model_panel.to_parquet(
    OUTPUT_FILES["model_panel"],
    index=False,
)

quality_report.to_csv(
    AUDIT_DIR / "notebook_01_quality_report.csv",
    index=False,
)

verification_status = (
    "READY_FOR_FULL_DOWNLOAD"
    if blocking_failures.empty
    and RUN_MODE == "AUDIT"
    else "READY_FOR_MODELLING"
    if blocking_failures.empty
    and RUN_MODE == "FULL"
    else "NOT_READY"
)

verification_report = {
    "verification_status": verification_status,
    "run_mode": RUN_MODE,
    "start_local": START_LOCAL.isoformat(),
    "end_local_exclusive": END_LOCAL.isoformat(),
    "expected_target_rows": EXPECTED_TARGET_ROWS,
    "actual_wind_rows": len(actual_wind),
    "day_ahead_rows": len(day_ahead_prices),
    "imbalance_native_rows": len(imbalance_native),
    "energinet_forecast_rows": len(
        energinet_wind_forecast
    ),
    "weather_point_rows": len(
        weather_point_forecasts
    ),
    "weather_area_rows": len(weather_area_long),
    "model_panel_shape": list(model_panel.shape),
    "weather_models_successful": (
        sorted(
            weather_area_long[
                "model_alias"
            ].unique().tolist()
        )
        if not weather_area_long.empty
        else []
    ),
    "weather_download_errors": weather_errors,
    "energinet_forecast_error": (
        energinet_forecast_error
    ),
    "blocking_failures": (
        blocking_failures.to_dict(orient="records")
    ),
    "output_files": {
        key: {
            "path": str(path),
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if path.exists()
                else None
            ),
        }
        for key, path in OUTPUT_FILES.items()
    },
}

with (
    AUDIT_DIR
    / "notebook_01_verification.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        verification_report,
        file,
        indent=2,
        default=str,
    )

print("=== NOTEBOOK 01 VERIFICATION ===")
print(
    json.dumps(
        verification_report,
        indent=2,
        default=str,
    )
)
print("=== END VERIFICATION ===")


## Acceptance gate

Proceed to Notebook 02 only when the verification status is:

- `READY_FOR_FULL_DOWNLOAD` after the audit run; then change `RUN_MODE` to `"FULL"` and rerun;
- `READY_FOR_MODELLING` after the full run.

A failed optional Energinet forecast benchmark does not block the core model dataset. Fewer than three successful exact weather models, missing targets, missing prices or information-time violations do block progression.
